# Spark Window Functions

## Basic to intermediate DataFrame patterns

Window functions calculate a value from a group of related rows **without collapsing those rows**. This makes them useful for ranking, running totals, comparisons with previous rows, and group-level statistics.

This notebook uses the PySpark DataFrame API and progresses from window specifications to practical analytical patterns.

## Learning objectives

By the end of this lesson, you should be able to:

- Explain the roles of `partitionBy`, `orderBy`, and a window frame.
- Use ranking functions: `row_number`, `rank`, `dense_rank`, `percent_rank`, and `ntile`.
- Use distribution functions: `cume_dist`.
- Compare rows with `lag` and `lead`.
- Retrieve ordered values with `first_value`, `last_value`, and `nth_value`.
- Apply aggregate functions over whole, cumulative, rolling, and value-range windows.
- Recognize correctness and performance issues involving ties, nulls, frames, shuffles, and sorting.

## 1. Start Spark before running the notebook

This notebook uses the same current Spark setup introduced in `D30_SparkDF`: a single-node Spark **standalone cluster**. It deliberately does not use the older notebook's `findspark` or `local[4]` configuration. In a WSL terminal, start the master and worker first:

```bash
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh spark://$(hostname):7077
```

Set `SPARK_MASTER` when the master URL differs from the hostname-based default.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D32-Spark-Window-Functions")
    .master(master_url)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

## 2. Create a small employee dataset

The data intentionally contains salary ties and a null bonus. A stable `employee_id` will be used as a tie-breaker where row order must be deterministic.

In [ ]:
employee_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("employee_name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("bonus", IntegerType(), True),
    StructField("join_date", StringType(), False),
])

employee_rows = [
    (101, "Anika",  "Engineering", 90000, 9000,  "2022-01-10"),
    (102, "Bala",   "Engineering", 78000, None,  "2023-03-15"),
    (103, "Chen",   "Engineering", 78000, 6000,  "2023-07-01"),
    (104, "Divya",  "Engineering", 65000, 4000,  "2024-02-20"),
    (201, "Farah",  "Finance",     85000, 8000,  "2021-11-05"),
    (202, "Gopal",  "Finance",     72000, 5000,  "2023-01-12"),
    (203, "Harini", "Finance",     72000, None,  "2023-09-18"),
    (204, "Ivan",   "Finance",     60000, 3500,  "2024-05-22"),
    (301, "Jaya",   "Sales",       82000, 12000, "2022-06-30"),
    (302, "Kiran",  "Sales",       70000, 10000, "2023-04-08"),
    (303, "Leela",  "Sales",       70000, 7000,  "2023-12-11"),
    (304, "Manoj",  "Sales",       58000, None,  "2024-06-17"),
]

employees = (
    spark.createDataFrame(employee_rows, employee_schema)
    .withColumn("join_date", F.to_date("join_date"))
)
employees.orderBy("department", F.desc("salary"), "employee_id").show(truncate=False)

## 3. Window anatomy

A window specification has up to three parts:

1. **Partition:** which rows belong to the same analytical group.
2. **Order:** the sequence inside each group.
3. **Frame:** which rows around the current row participate in the calculation.

```text
partitionBy(department) -> orderBy(salary) -> rowsBetween(start, end)
```

Unlike `groupBy`, a window calculation preserves one output row for every input row. Most partitioned and ordered windows require a shuffle followed by a sort.

In [ ]:
department_window = Window.partitionBy("department")
salary_rank_window = Window.partitionBy("department").orderBy(F.desc("salary"))
deterministic_row_window = Window.partitionBy("department").orderBy(
    F.desc("salary"), F.asc("employee_id")
)

## 4. `row_number`: a unique sequence

`row_number` assigns `1, 2, 3, ...` inside each partition. Because tied salaries do not define an order by themselves, add a stable tie-breaker when the exact row numbers matter.

In [ ]:
(employees
 .withColumn("row_number", F.row_number().over(deterministic_row_window))
 .orderBy("department", "row_number")
 .show())

## 5. `rank` and `dense_rank`: ties

Both functions give tied salaries the same position. `rank` leaves a gap after a tie (`1, 2, 2, 4`), while `dense_rank` does not (`1, 2, 2, 3`). Keep only the business ordering column here; adding `employee_id` would break salary ties.

In [ ]:
ranking_result = (
    employees
    .withColumn("rank", F.rank().over(salary_rank_window))
    .withColumn("dense_rank", F.dense_rank().over(salary_rank_window))
)
ranking_result.orderBy("department", "rank", "employee_id").show()

## 6. `percent_rank`: position based on rank

Start with four scores ordered from highest to lowest: `100, 90, 90, 70`.

`percent_rank` uses this formula:

```text
(rank - 1) / (number of rows - 1)
```

| Score | Rank | Calculation | percent_rank |
| ---: | ---: | --- | ---: |
| 100 | 1 | (1 - 1) / 3 | 0.000 |
| 90 | 2 | (2 - 1) / 3 | 0.333 |
| 90 | 2 | (2 - 1) / 3 | 0.333 |
| 70 | 4 | (4 - 1) / 3 | 1.000 |

The tied scores receive the same rank and therefore the same percentage. The first row is always `0.0`; in a partition containing only one row, Spark also returns `0.0`.

In [ ]:
score_df = spark.createDataFrame([(100,), (90,), (90,), (70,)], ["score"])
score_window = Window.orderBy(F.desc("score"))

(score_df
 .withColumn("rank", F.rank().over(score_window))
 .withColumn("percent_rank", F.round(F.percent_rank().over(score_window), 3))
 .orderBy(F.desc("score"))
 .show())

## 7. `cume_dist`: cumulative proportion

Use the same four descending scores: `100, 90, 90, 70`.

`cume_dist` uses this idea:

```text
number of rows at or before the current score / total number of rows
```

Because the order is descending, **at or before** means scores greater than or equal to the current score.

| Score | Rows at or before it | Calculation | cume_dist |
| ---: | ---: | --- | ---: |
| 100 | 1 | 1 / 4 | 0.25 |
| 90 | 3 | 3 / 4 | 0.75 |
| 90 | 3 | 3 / 4 | 0.75 |
| 70 | 4 | 4 / 4 | 1.00 |

Both tied 90s receive `0.75` because the cumulative position moves to the end of the complete tie group. Unlike `percent_rank`, the first `cume_dist` value is normally greater than zero.

In [ ]:
(score_df
 .withColumn("cume_dist", F.round(F.cume_dist().over(score_window), 2))
 .orderBy(F.desc("score"))
 .show())

## 8. `ntile`: place rows into buckets

`ntile(n)` assigns ordered rows to `n` buckets whose sizes differ by at most one. It is useful for quartiles or bands, but small partitions may have empty conceptual buckets. A deterministic order makes tied rows reproducible.

In [ ]:
(employees
 .withColumn("salary_quartile", F.ntile(4).over(deterministic_row_window))
 .orderBy("department", "salary_quartile", F.desc("salary"))
 .show())

## 9. `lag` and `lead`: compare nearby rows

`lag` reads an earlier row and `lead` reads a later row in the window order. They are commonly used for change calculations. The optional third argument is the default used when no such row exists.

In [ ]:
salary_comparison = (
    employees
    .withColumn("previous_salary", F.lag("salary", 1).over(deterministic_row_window))
    .withColumn("next_salary", F.lead("salary", 1).over(deterministic_row_window))
    .withColumn("difference_from_previous", F.col("salary") - F.col("previous_salary"))
)
salary_comparison.orderBy("department", F.desc("salary"), "employee_id").show()

## 10. `first_value`, `last_value`, and `nth_value`

These functions retrieve a value from an ordered window. The frame is crucial: with an ordered window, the default frame usually ends at the current row, so `last_value` may simply return the current value. Define the entire-partition frame when you mean the final value in the group. `nth_value` uses a 1-based position.

In [ ]:
whole_department_by_salary = (
    Window.partitionBy("department")
    .orderBy(F.desc("salary"), F.asc("employee_id"))
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

value_result = (
    employees
    .withColumn("highest_paid_employee", F.first_value("employee_name").over(whole_department_by_salary))
    .withColumn("lowest_paid_employee", F.last_value("employee_name").over(whole_department_by_salary))
    .withColumn("second_ordered_salary", F.nth_value("salary", 2).over(whole_department_by_salary))
)
value_result.orderBy("department", F.desc("salary"), "employee_id").show()

### Null handling in value functions

Spark 3.5 supports `ignoreNulls=True` for `first_value` and `last_value`, and `ignoreNulls=True` as the third argument of `nth_value`. This finds the first non-null bonus in join-date order.

In [ ]:
whole_department_by_join_date = (
    Window.partitionBy("department")
    .orderBy("join_date", "employee_id")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

(employees
 .withColumn(
     "first_recorded_bonus",
     F.first_value("bonus", ignoreNulls=True).over(whole_department_by_join_date),
 )
 .orderBy("department", "join_date")
 .show())

## 11. Aggregate functions over a partition

Standard aggregates such as `count`, `sum`, `avg`, `min`, and `max` become window functions when followed by `.over(...)`. With only `partitionBy`, every row sees the complete department.

In [ ]:
department_statistics = (
    employees
    .withColumn("department_size", F.count("*").over(department_window))
    .withColumn("department_payroll", F.sum("salary").over(department_window))
    .withColumn("department_avg_salary", F.round(F.avg("salary").over(department_window), 2))
    .withColumn("department_min_salary", F.min("salary").over(department_window))
    .withColumn("department_max_salary", F.max("salary").over(department_window))
    .withColumn("salary_minus_department_avg", F.round(
        F.col("salary") - F.avg("salary").over(department_window), 2
    ))
)
department_statistics.orderBy("department", F.desc("salary")).show()

## 12. Row frames: cumulative and rolling calculations

`rowsBetween` selects rows by physical position in the order:

- `unboundedPreceding` to `currentRow`: all rows from the start through the current row.
- `-1` to `currentRow`: the current row and one preceding row.
- `unboundedPreceding` to `unboundedFollowing`: the complete partition.

In [ ]:
join_order = Window.partitionBy("department").orderBy("join_date", "employee_id")
cumulative_frame = join_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
two_row_frame = join_order.rowsBetween(-1, Window.currentRow)

frame_result = (
    employees
    .withColumn("cumulative_payroll", F.sum("salary").over(cumulative_frame))
    .withColumn("two_employee_moving_avg", F.round(F.avg("salary").over(two_row_frame), 2))
)
frame_result.orderBy("department", "join_date", "employee_id").show()

## 13. Range frames: calculations by ordered value

`rangeBetween` uses the value of the ordering expression rather than a row count. With salary as the single numeric ordering expression, `rangeBetween(-10000, 0)` includes salaries from 10,000 below the current salary through the current salary. Peer rows with the same salary are included together.

In [ ]:
salary_band_window = (
    Window.partitionBy("department")
    .orderBy(F.col("salary"))
    .rangeBetween(-10000, 0)
)

(employees
 .withColumn("employees_in_10k_band", F.count("*").over(salary_band_window))
 .withColumn("average_in_10k_band", F.round(F.avg("salary").over(salary_band_window), 2))
 .orderBy("department", "salary", "employee_id")
 .show())

## 14. Practical pattern: top N per group

Use `row_number` when exactly N rows are required. Use `dense_rank` when all ties at the Nth business value must be retained. The two answers can contain different row counts.

In [ ]:
top_two_exactly = (
    employees
    .withColumn("position", F.row_number().over(deterministic_row_window))
    .filter(F.col("position") <= 2)
)

top_two_salary_levels_with_ties = (
    employees
    .withColumn("salary_level", F.dense_rank().over(salary_rank_window))
    .filter(F.col("salary_level") <= 2)
)

print("Exactly two employees per department")
top_two_exactly.orderBy("department", "position").show()
print("Top two salary levels, including ties")
top_two_salary_levels_with_ties.orderBy("department", "salary_level", "employee_id").show()

## 15. Practical pattern: contribution to group total

Dividing each salary by the department payroll produces a ratio-to-report calculation. Spark has no special function requirement here; combine `sum` over a window with a normal column expression.

In [ ]:
(employees
 .withColumn("department_payroll", F.sum("salary").over(department_window))
 .withColumn(
     "payroll_share_pct",
     F.round(F.col("salary") / F.col("department_payroll") * 100, 2),
 )
 .orderBy("department", F.desc("salary"))
 .show())

## 16. Practical pattern: deduplication

A common data-engineering use case is to retain the latest record for each business key. Partition by the key, order newest first, assign `row_number`, and keep row 1. Include a deterministic tie-breaker if timestamps can be equal.

In [ ]:
status_updates = spark.createDataFrame([
    (101, "ACTIVE",   "2026-08-25 09:00:00", 1),
    (101, "ON_LEAVE", "2026-08-27 14:30:00", 2),
    (202, "ACTIVE",   "2026-08-26 10:15:00", 1),
    (202, "INACTIVE", "2026-08-28 08:00:00", 2),
], ["employee_id", "status", "updated_at_text", "event_id"])

status_updates = status_updates.withColumn(
    "updated_at", F.to_timestamp("updated_at_text")
).drop("updated_at_text")

latest_window = Window.partitionBy("employee_id").orderBy(
    F.desc("updated_at"), F.desc("event_id")
)

latest_status = (
    status_updates
    .withColumn("row_number", F.row_number().over(latest_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)
latest_status.orderBy("employee_id").show(truncate=False)

## 17. Inspect the execution plan

A partitioned window commonly introduces an `Exchange` to colocate keys and a `Sort` to establish the required order. Reuse identical window specifications in one projection where practical so Spark can often evaluate several expressions together.

In [ ]:
(employees
 .withColumn("rank", F.rank().over(salary_rank_window))
 .withColumn("dense_rank", F.dense_rank().over(salary_rank_window))
 .explain(mode="formatted"))

## 18. Correctness and performance checklist

- Use a tie-breaker for deterministic `row_number`, `lag`, `lead`, and positional value results.
- Do not add a tie-breaker to `rank` or `dense_rank` when business values should remain tied.
- Specify the frame explicitly for `last_value`, cumulative calculations, and rolling calculations.
- Remember that `rowsBetween` counts row positions; `rangeBetween` compares ordering values.
- Avoid a window without `partitionBy` on large data unless a single global partition is truly intended.
- Expect shuffling and sorting; watch for skew when one partition key owns far more rows than others.
- Prefer built-in Spark functions to Python UDFs for window expressions.
- Apply filters before the window only when doing so matches the business meaning; filtering changes which rows the window can see.
- Use `count("*")` to count rows; `count("nullable_column")` counts only non-null values.

## 19. Practice exercises

1. Return the three lowest-paid employees in each department with exactly three rows per department.
2. Calculate each employee's salary difference from the next lower salary.
3. Show a three-row moving average ordered by join date.
4. Find every employee belonging to the highest two distinct salary levels in each department.
5. For each department, show the earliest joiner and the latest joiner on every row.

In [ ]:
# Write the practice solutions here.

## Summary

Window functions preserve row-level detail while adding information derived from related rows. Choose the partition for the business group, the ordering for sequence or rank, and the frame for the exact neighborhood. Ranking, offset, value, distribution, and aggregate functions can then solve a wide range of analytical and data-engineering problems with DataFrame expressions.

## Stop Spark

Run this cell when the lesson is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")